# Tugas Evaluasi CPMK 3 — Evaluasi Test Set Custom & Domain Shift

Notebook ini **tidak melatih model**. Isinya memuat 9 bobot hasil `CPMK3_training.ipynb`, lalu mengevaluasinya di **test set buatan sendiri** (41 foto HP, 10 kondisi sulit) dan membandingkannya dengan test set asli.

**Input yang perlu ditambahkan di Kaggle** (*Add Input*, panel kanan):
1. **Notebook Output** dari versi commit `CPMK3_training` (berisi `CPMK3/runs`, `CPMK3/results`, `CPMK3/preds`).
2. **Dataset** yang kamu upload, berisi: folder gambar export Roboflow (41 `.jpg`), `test_custom_clean.json` (hasil `clean_custom.py`), dan `conditions.csv`.

Pengaturan lainnya sama dengan notebook training: GPU T4, Internet On, Secrets `ROBOFLOW_API_KEY` (test set asli tetap di-download untuk pemetaan kelas dan pembanding).

**Supaya perbandingan adil**
1. **Kelas sama.** Test custom hanya dianotasi untuk kelas objek (kelas dinding/lantai/plafon dikecualikan). Karena itu test asli juga dihitung ulang **hanya dengan kelas yang ada di test custom** → split `test_sub`. Perbandingan domain shift = `test_sub` vs `test_custom`.
2. **Resolusi sama.** Foto HP (12 MP) di-resize ke sisi terpanjang = median sisi terpanjang test asli, dan anotasinya ikut di-skala. Tanpa ini APs/APm/APl tidak sebanding (batas small/medium/large memakai piksel absolut) dan inferensi 12 MP bisa OOM.
3. Fungsi prediksi dan evaluasi **sama persis** dengan notebook training (sel di bawah disalin dari generator yang sama).

## 1. Setup

In [ ]:
!nvidia-smi
!pip -q install ultralytics roboflow pycocotools

In [ ]:
import os, sys, json, time, math, random, csv, glob, datetime, contextlib, io, copy
from collections import defaultdict
import numpy as np, cv2, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageOps
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils

import gc, traceback
IN_COLAB  = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle/working')

# ===================== KONFIGURASI UTAMA =====================
NRP_LAST_DIGIT = 3          # digit 3 -> variabel: jumlah data latih (25% / 50% / 100%)
FRACTIONS      = [0.25, 0.5, 1.0]                     # variabel eksperimen (NRP ...3)
MODELS_TO_RUN  = ['yolo', 'maskrcnn', 'mask2former']  # bisa dikurangi kalau mau run sebagian
QUICK_TEST     = False      # True = uji coba cepat (budget 2 menit/run) untuk cek error sebelum commit
IMG_SIZE       = 640        # sisi terpanjang input, sama untuk semua model
SEED           = 42
TIME_BUDGET_H  = 0.66       # batas waktu training PER RUN (jam). 3 fraksi x 0.66 jam ≈ 2 jam per model
MAX_EPOCHS     = 100        # batas atas; biasanya yang menghentikan training adalah TIME_BUDGET_H
if QUICK_TEST: TIME_BUDGET_H, MAX_EPOCHS = 0.03, 2
TRAIN_FRACTION = FRACTIONS[-1]   # diatur otomatis oleh set_fraction()

RF_WORKSPACE, RF_PROJECT, RF_VERSION = "panopticindoor", "panoptic-indoor-segmentation", 5

if IN_KAGGLE:   # /kaggle/working = output tersimpan (maks 20 GB); data mentah taruh di /tmp agar tidak ikut tersimpan
    OUT_ROOT, DATA_ROOT = "/kaggle/working/CPMK3", "/tmp/cpmk3_data"
elif IN_COLAB:
    OUT_ROOT, DATA_ROOT = "/content/drive/MyDrive/CPMK3", "/content/data"
else:
    OUT_ROOT, DATA_ROOT = os.path.abspath("cpmk3_out"), os.path.abspath("cpmk3_data")
if QUICK_TEST: OUT_ROOT += "_quicktest"
# =============================================================

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
print('device:', DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '', '| torch', torch.__version__)
print('output ->', OUT_ROOT)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
for d in [OUT_ROOT, f"{OUT_ROOT}/runs", f"{OUT_ROOT}/results", f"{OUT_ROOT}/preds", DATA_ROOT]:
    os.makedirs(d, exist_ok=True)

### 1.1 Logbook & pencatat waktu GPU (bukti proses, poin 4)
- `gpu_time_log.csv`: tiap run training otomatis tercatat (GPU, mulai, selesai, menit, peak memory, status).
- `LOGBOOK.md`: catat kendala dan keputusan dengan `log_note("...")`. Contoh: `log_note("YOLO OOM di batch 32, turun ke 16")`.

In [ ]:
GPU_LOG = f"{OUT_ROOT}/gpu_time_log.csv"
LOGBOOK = f"{OUT_ROOT}/LOGBOOK.md"

def log_note(text):
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    with open(LOGBOOK, 'a') as f:
        f.write(f"- **{ts}** — {text}\n")
    print('logbook +', text)

class GPUTimer:
    # Dipakai sebagai: with GPUTimer(run, model) as t: ...  -> t.minutes
    def __init__(self, run, model, note=''):
        self.run, self.model, self.note, self.minutes = run, model, note, None
    def __enter__(self):
        self.start, self.t0 = datetime.datetime.now(), time.time()
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        return self
    def __exit__(self, exc_type, exc, tb):
        self.minutes = (time.time() - self.t0) / 60
        cuda = torch.cuda.is_available()
        gpu  = torch.cuda.get_device_name(0) if cuda else 'cpu'
        peak = torch.cuda.max_memory_allocated() / 1e9 if cuda else 0.0
        status = 'ok' if exc_type is None else f'stopped: {exc_type.__name__}'
        new = not os.path.exists(GPU_LOG)
        with open(GPU_LOG, 'a', newline='') as f:
            w = csv.writer(f)
            if new: w.writerow(['run','model','gpu','start','end','minutes','peak_mem_gb','status','note'])
            w.writerow([self.run, self.model, gpu, self.start.isoformat(timespec='seconds'),
                        datetime.datetime.now().isoformat(timespec='seconds'),
                        f"{self.minutes:.2f}", f"{peak:.2f}", status, self.note])
        print(f"⏱  {self.run}: {self.minutes:.1f} menit di {gpu} (peak {peak:.1f} GB) — {status}")
        return False   # error tetap dilempar

def gpu_minutes(run):
    # total menit GPU sebuah run dari log (berguna kalau sesi Colab sudah restart)
    if not os.path.exists(GPU_LOG): return None
    df = pd.read_csv(GPU_LOG)
    return float(df[df.run == run].minutes.sum()) or None

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

## 2. Dataset

### 2.1 Download dari Roboflow (format COCO segmentation)
API key dibaca dari Kaggle Secrets / Colab Secrets dengan nama `ROBOFLOW_API_KEY`.

In [ ]:
COCO_DIR = f"{DATA_ROOT}/coco"
if not os.path.exists(f"{COCO_DIR}/train/_annotations.coco.json"):
    from roboflow import Roboflow
    RF_KEY = os.environ.get('ROBOFLOW_API_KEY')
    if not RF_KEY and IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        RF_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
    if not RF_KEY and IN_COLAB:
        from google.colab import userdata
        RF_KEY = userdata.get('ROBOFLOW_API_KEY')
    if not RF_KEY:
        import getpass; RF_KEY = getpass.getpass('Roboflow API key: ')
    rf = Roboflow(api_key=RF_KEY)
    version = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
    # overwrite=True: tanpa ini roboflow DIAM-DIAM melewati download kalau folder sudah ada (mis. sisa run gagal)
    try:
        version.download("coco-segmentation", location=COCO_DIR, overwrite=True)
    except Exception as e:
        print("format coco-segmentation gagal:", repr(e)[:200], "-> coba format 'coco'")
        version.download("coco", location=COCO_DIR, overwrite=True)

# cari file anotasi di mana pun di dalam COCO_DIR (antisipasi struktur folder berbeda)
ann_files = sorted(glob.glob(f"{COCO_DIR}/**/_annotations.coco.json", recursive=True))
IMG_DIRS = {os.path.basename(os.path.dirname(p)).lower(): os.path.dirname(p) for p in ann_files}
if 'val' in IMG_DIRS and 'valid' not in IMG_DIRS: IMG_DIRS['valid'] = IMG_DIRS.pop('val')
SPLITS = [s for s in ['train', 'valid', 'test'] if s in IMG_DIRS]
print('split tersedia:', {s: IMG_DIRS[s] for s in SPLITS})
if not SPLITS:
    print("Isi COCO_DIR (untuk diagnosis):")
    for root, dirs, files in os.walk(COCO_DIR):
        print(' ', root, '|', len(files), 'file', files[:3])
    raise RuntimeError("File _annotations.coco.json tidak ditemukan. Cek output di atas / API key / akses internet.")
assert 'train' in SPLITS and 'test' in SPLITS, f"Split kurang: {SPLITS} — catat di logbook dan buat split dari train/valid."

### 2.2 Bersihkan anotasi
Export COCO dari Roboflow biasanya punya satu kategori "induk" (superkategori) yang tidak dipakai anotasi mana pun, jadi kategori itu dibuang. Kelas = kategori yang benar-benar muncul di train. Anotasi tanpa polygon valid (kurang dari 3 titik) juga dibuang. Hasilnya ditulis ke `clean/{split}.json`, dan semua model membaca file ini.

In [ ]:
CLEAN_DIR = f"{DATA_ROOT}/clean"; os.makedirs(CLEAN_DIR, exist_ok=True)
raw = {s: json.load(open(f"{IMG_DIRS[s]}/_annotations.coco.json")) for s in SPLITS}

cats_by_id  = {c['id']: c for c in raw['train']['categories']}
name2id     = {c['name']: c['id'] for c in raw['train']['categories']}
CAT_IDS     = sorted({a['category_id'] for a in raw['train']['annotations']})
CLASS_NAMES = [cats_by_id[i]['name'] for i in CAT_IDS]
NUM_CLASSES = len(CAT_IDS)
CATID2IDX   = {cid: i for i, cid in enumerate(CAT_IDS)}   # id COCO -> indeks 0..K-1
print(f"{NUM_CLASSES} kelas:", CLASS_NAMES)
print("kategori dibuang (tanpa anotasi di train):", [c['name'] for c in raw['train']['categories'] if c['id'] not in CATID2IDX])

def clean_split(js):
    local = {c['id']: c['name'] for c in js['categories']}
    anns, dropped = [], defaultdict(int)
    for a in js['annotations']:
        cid = name2id.get(local.get(a['category_id']))          # samakan id antar split lewat nama
        if cid not in CATID2IDX: dropped['kelas tidak ada di train'] += 1; continue
        seg = a.get('segmentation')
        if isinstance(seg, list):
            seg = [p for p in seg if len(p) >= 6]
            if not seg: dropped['polygon kosong/<3 titik'] += 1; continue
        elif not seg: dropped['tanpa segmentasi'] += 1; continue
        a = dict(a, category_id=cid, segmentation=seg)
        a.setdefault('iscrowd', 0)
        a.setdefault('area', float(a['bbox'][2] * a['bbox'][3]))
        anns.append(a)
    out = {'info': js.get('info', {}), 'images': js['images'], 'annotations': anns,
           'categories': [cats_by_id[i] for i in CAT_IDS]}
    return out, dict(dropped)

for s in SPLITS:
    out, dropped = clean_split(raw[s])
    json.dump(out, open(f"{CLEAN_DIR}/{s}.json", 'w'))
    print(f"{s:6s}: {len(out['images']):5d} gambar, {len(out['annotations']):6d} instance, dibuang: {dropped or '-'}")

### 2.3 EDA singkat
Jumlah instance per kelas dan proporsi objek kecil/sedang/besar (definisi COCO: kecil < 32², besar > 96² piksel). Bagian ini berguna nanti untuk menjelaskan kenapa AP tiap kelas berbeda.

In [ ]:
rows = []
for s in SPLITS:
    js = json.load(open(f"{CLEAN_DIR}/{s}.json"))
    for a in js['annotations']:
        area = a['area']
        rows.append(dict(split=s, cls=CLASS_NAMES[CATID2IDX[a['category_id']]],
                         size='small' if area < 32**2 else ('large' if area > 96**2 else 'medium')))
eda = pd.DataFrame(rows)
cls_table = eda.pivot_table(index='cls', columns='split', aggfunc='size', fill_value=0)
cls_table = cls_table[[s for s in SPLITS if s in cls_table.columns]].sort_values('train', ascending=False)
display(cls_table)
display(pd.crosstab(eda.cls, eda['size'], normalize='index').round(2).loc[cls_table.index])

ax = cls_table['train'].plot.barh(figsize=(7, max(3, 0.3 * NUM_CLASSES)), title='Instance per kelas (train)')
ax.invert_yaxis(); plt.tight_layout(); plt.show()

In [ ]:
def load_image(path):
    img = Image.open(path)
    img = ImageOps.exif_transpose(img)   # samakan orientasi dengan cv2.imread (dipakai Ultralytics)
    return img.convert('RGB')

PALETTE = np.random.RandomState(0).randint(40, 255, (256, 3))

def draw_instances(img, masks, labels, scores=None, ax=None, title=''):
    # img: PIL/np RGB; masks: list/array (N,H,W) bool; labels: indeks 0..K-1
    im = np.array(img).astype(np.float32)
    for i, (m, l) in enumerate(zip(masks, labels)):
        im[m] = 0.5 * im[m] + 0.5 * PALETTE[i % 256]
    ax = ax or plt.gca()
    ax.imshow(im.astype(np.uint8)); ax.set_title(title, fontsize=9); ax.axis('off')
    for i, (m, l) in enumerate(zip(masks, labels)):
        ys, xs = np.nonzero(m)
        if len(xs):
            txt = CLASS_NAMES[l] + (f" {scores[i]:.2f}" if scores is not None else '')
            ax.text(xs.mean(), ys.mean(), txt, fontsize=6, color='white', bbox=dict(facecolor='black', alpha=0.5, pad=1))

def show_gt(split, n=4, seed=0):
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(f"{CLEAN_DIR}/{split}.json")
    ids = random.Random(seed).sample(sorted(coco.imgs), min(n, len(coco.imgs)))
    fig, axs = plt.subplots(1, len(ids), figsize=(5 * len(ids), 5))
    for ax, i in zip(np.atleast_1d(axs), ids):
        info = coco.imgs[i]; anns = coco.loadAnns(coco.getAnnIds(imgIds=i, iscrowd=False))
        draw_instances(load_image(os.path.join(IMG_DIRS[split], info['file_name'])),
                       [coco.annToMask(a).astype(bool) for a in anns],
                       [CATID2IDX[a['category_id']] for a in anns], ax=ax, title=info['file_name'][:30])
    plt.tight_layout(); plt.show()

show_gt('train')

### 2.4 Subset data latih (variabel NRP)
Gambar train yang punya anotasi diacak **sekali** dengan `SEED`, lalu diambil prefiks sebesar `TRAIN_FRACTION`. Karena urutannya sama, subset 25% ⊂ 50% ⊂ 100%. Validasi dan test **tidak** dikurangi.

Perhatikan output distribusi kelas: di 25%, kelas langka bisa tinggal sedikit sekali contohnya. Ini bahan analisis yang bagus.

In [ ]:
def make_train_subset(frac, seed=SEED):
    js = json.load(open(f"{CLEAN_DIR}/train.json"))
    has_ann = sorted({a['image_id'] for a in js['annotations'] if not a.get('iscrowd', 0)})
    order = has_ann[:]; random.Random(seed).shuffle(order)
    keep = set(order[:max(1, int(round(frac * len(order))))])
    sub = dict(js, images=[im for im in js['images'] if im['id'] in keep],
               annotations=[a for a in js['annotations'] if a['image_id'] in keep])
    path = f"{CLEAN_DIR}/train_f{int(round(frac * 100))}.json"
    json.dump(sub, open(path, 'w'))
    cnt = pd.Series([CLASS_NAMES[CATID2IDX[a['category_id']]] for a in sub['annotations']]).value_counts()
    print(f"subset {frac:.0%}: {len(sub['images'])} gambar, {len(sub['annotations'])} instance -> {path}")
    print("kelas paling sedikit:", cnt.tail(5).to_dict())
    return path

def set_fraction(frac):
    # ganti fraksi aktif: semua fungsi run_* membaca TRAIN_FRACTION & TRAIN_JSON global
    global TRAIN_FRACTION, TRAIN_JSON
    TRAIN_FRACTION, TRAIN_JSON = frac, make_train_subset(frac)

for f in FRACTIONS: set_fraction(f)      # tampilkan distribusi kelas tiap subset
VAL_JSON   = f"{CLEAN_DIR}/valid.json"
TEST_JSON  = f"{CLEAN_DIR}/test.json"

## 3. Evaluasi bersama (COCO mask mAP)
Semua model mengeluarkan prediksi dalam format hasil COCO (`image_id, category_id, segmentation RLE, score`), lalu dinilai dengan fungsi yang sama:
- **AP** = mAP@[.50:.95] (metrik utama), **AP50**, **AP75**, **APs/APm/APl** (per ukuran objek)
- **AP per kelas**, dipakai untuk analisis kelas sulit dan analisis kegagalan

Prediksi disimpan ke Drive (`preds/`), jadi visualisasi kegagalan nanti bisa dilakukan tanpa inferensi ulang.

In [ ]:
import torchvision.transforms.functional as TF

def load_instances(coco, img_id):
    # -> masks (N,H,W) uint8, labels 0..K-1; buang mask kosong
    masks, labels = [], []
    for a in coco.loadAnns(coco.getAnnIds(imgIds=img_id, iscrowd=False)):
        m = coco.annToMask(a)
        if m.any(): masks.append(m); labels.append(CATID2IDX[a['category_id']])
    H, W = coco.imgs[img_id]['height'], coco.imgs[img_id]['width']
    return (np.stack(masks) if masks else np.zeros((0, H, W), np.uint8)), np.array(labels, np.int64)

def mask_to_rle(m):
    rle = mask_utils.encode(np.asfortranarray(m.astype(np.uint8)))
    rle['counts'] = rle['counts'].decode('ascii')
    return rle

METRIC_NAMES = ['AP', 'AP50', 'AP75', 'APs', 'APm', 'APl']

def coco_eval_segm(gt_json, results, verbose=True):
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(gt_json)
    if len(results) == 0:
        return {**{k: 0.0 for k in METRIC_NAMES}, 'per_class': {n: 0.0 for n in CLASS_NAMES}}
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        dt = gt.loadRes(copy.deepcopy(results))     # loadRes memodifikasi list (area numpy) -> salin
        E = COCOeval(gt, dt, 'segm'); E.evaluate(); E.accumulate(); E.summarize()
    if verbose: print('\n'.join(l for l in buf.getvalue().splitlines() if l.startswith(' Average')))
    out = {k: float(E.stats[i]) for i, k in enumerate(METRIC_NAMES)}
    prec = E.eval['precision']              # [IoU, recall, kelas, area, maxDet]
    out['per_class'] = {}
    for k, cid in enumerate(E.params.catIds):
        p = prec[:, :, k, 0, -1]; p = p[p > -1]
        out['per_class'][cats_by_id[cid]['name']] = float(p.mean()) if p.size else float('nan')
    return out

def run_name(model_key):
    return f"{model_key}_f{int(round(TRAIN_FRACTION * 100))}_s{IMG_SIZE}"

def already_done(run, split='test'):
    done = os.path.exists(f"{OUT_ROOT}/results/{run}__{split}.json")
    if done: print(f"⏭  {run} sudah ada hasilnya, dilewati (hapus file results-nya untuk mengulang)")
    return done

def save_run(run, model_name, family, eval_split, metrics, preds, extra=None):
    rec = dict(run=run, model=model_name, family=family, eval_split=eval_split,
               train_fraction=TRAIN_FRACTION, img_size=IMG_SIZE, seed=SEED,
               gpu_minutes=gpu_minutes(run), n_preds=len(preds),
               timestamp=datetime.datetime.now().isoformat(timespec='seconds'),
               **{k: v for k, v in metrics.items() if k != 'per_class'},
               per_class=metrics['per_class'], **(extra or {}))
    json.dump(rec, open(f"{OUT_ROOT}/results/{run}__{eval_split}.json", 'w'), indent=2)
    json.dump(preds, open(f"{OUT_ROOT}/preds/{run}__{eval_split}.json", 'w'))
    print(f"✅ {run} @ {eval_split}: AP={rec['AP']:.4f} AP50={rec['AP50']:.4f}  (disimpan)")
    return rec

def show_preds(run, eval_split, gt_json, img_dir, n=3, score_thr=0.5, img_ids=None, seed=0):
    # GT (kiri) vs prediksi (kanan) dari file prediksi tersimpan
    preds = json.load(open(f"{OUT_ROOT}/preds/{run}__{eval_split}.json"))
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    img_ids = img_ids or random.Random(seed).sample(sorted(coco.imgs), min(n, len(coco.imgs)))
    by_img = defaultdict(list)
    for p in preds:
        if p['score'] >= score_thr: by_img[p['image_id']].append(p)
    fig, axs = plt.subplots(len(img_ids), 2, figsize=(11, 5 * len(img_ids)), squeeze=False)
    for row, i in zip(axs, img_ids):
        info = coco.imgs[i]; img = load_image(os.path.join(img_dir, info['file_name']))
        anns = coco.loadAnns(coco.getAnnIds(imgIds=i, iscrowd=False))
        draw_instances(img, [coco.annToMask(a).astype(bool) for a in anns],
                       [CATID2IDX[a['category_id']] for a in anns], ax=row[0], title=f"GT {info['file_name'][:25]}")
        ps = by_img[i]
        draw_instances(img, [mask_utils.decode(p['segmentation']).astype(bool) for p in ps],
                       [CATID2IDX[p['category_id']] for p in ps], [p['score'] for p in ps],
                       ax=row[1], title=f"{run} (score≥{score_thr})")
    plt.tight_layout(); plt.show()

### 3.1 Training loop generik (untuk Mask R-CNN & Mask2Former)
YOLO memakai trainer Ultralytics. Dua model lain memakai loop yang sama:
- **LR schedule**: warmup linear, lalu cosine. Progres dihitung dari `max(iterasi/total, waktu/budget)`, jadi LR tetap turun ke minimum walaupun training dihentikan oleh batas waktu, bukan oleh jumlah epoch. Ultralytics juga melakukan hal serupa saat `time=` dipakai.
- Validasi (COCO segm AP di `valid`) tiap `val_every` epoch, lalu simpan `best.pt` dan `last.pt` ke Drive.
- Log per epoch ke `results.csv` (bukti proses, poin 4).
- Step dengan loss NaN/inf dilewati dan dihitung; kalau banyak, matikan AMP.

In [ ]:
def train_loop(run, model, dl_train, step_fn, optimizer, eval_fn, budget_h=TIME_BUDGET_H,
               max_epochs=MAX_EPOCHS, val_every=1, amp=USE_AMP, clip=None, warmup_iters=300, min_lr_ratio=0.01):
    out_dir = f"{OUT_ROOT}/runs/{run}"; os.makedirs(out_dir, exist_ok=True)
    csv_path = f"{out_dir}/results.csv"
    if os.path.exists(csv_path): os.remove(csv_path)
    base_lrs = [g['lr'] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    t0, budget, total = time.time(), budget_h * 3600, max_epochs * len(dl_train)
    it, best_ap, skipped, stop = 0, -1.0, 0, False
    for ep in range(1, max_epochs + 1):
        model.train(); sums, n = defaultdict(float), 0
        for batch in dl_train:
            prog = max(it / total, (time.time() - t0) / budget)
            if prog >= 1: stop = True; break
            f = min(1.0, (it + 1) / warmup_iters) * (min_lr_ratio + (1 - min_lr_ratio) * 0.5 * (1 + math.cos(math.pi * prog)))
            for g, b in zip(optimizer.param_groups, base_lrs): g['lr'] = b * f
            with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=amp):
                loss, logs = step_fn(model, batch)
            it += 1
            if not torch.isfinite(loss):
                skipped += 1; optimizer.zero_grad(set_to_none=True); continue
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if clip:
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer); scaler.update()
            for k, v in logs.items(): sums[k] += v
            n += 1
            if it % 50 == 0:
                print(f"  ep{ep} it{it} loss={logs.get('loss', loss.item()):.3f} lr={optimizer.param_groups[0]['lr']:.2e} "
                      f"t={(time.time()-t0)/60:.1f}m", flush=True)
        if ep == max_epochs: stop = True
        row = {'epoch': ep, 'iters': it, 'minutes': round((time.time() - t0) / 60, 2),
               'lr': optimizer.param_groups[0]['lr'], 'skipped_steps': skipped,
               **{f"train/{k}": v / max(n, 1) for k, v in sums.items()},
               **{f"val/{k}": np.nan for k in METRIC_NAMES}}       # kolom tetap sama tiap baris
        if ep % val_every == 0 or stop:
            model.eval(); m = eval_fn(model)
            row.update({f"val/{k}": m[k] for k in METRIC_NAMES})
            if m['AP'] > best_ap:
                best_ap = m['AP']; torch.save(model.state_dict(), f"{out_dir}/best.pt")
            print(f"[ep {ep}] {row['minutes']} menit | val AP={m['AP']:.4f} AP50={m['AP50']:.4f} | best={best_ap:.4f}")
        pd.DataFrame([row]).to_csv(csv_path, mode='a', header=not os.path.exists(csv_path), index=False)
        if stop: break
    torch.save(model.state_dict(), f"{out_dir}/last.pt")
    if skipped > 0.1 * max(it, 1): print(f"⚠️ {skipped}/{it} step NaN — pertimbangkan amp=False")
    return f"{out_dir}/best.pt"

## 4. Model A — YOLO11s-seg (one-stage, CNN)

**Cara kerja singkat**: backbone CNN dan neck FPN/PAN menghasilkan prediksi padat di tiap grid untuk box, kelas, dan 32 koefisien mask. Ada cabang *Proto* yang menghasilkan 32 prototype mask berukuran **1/4 input** (160×160 untuk 640). Mask tiap objek = sigmoid(koefisien · prototype), di-crop ke box, lalu diperbesar. Duplikat dihapus dengan NMS.

Konsekuensi yang perlu diingat untuk analisis kegagalan:
- detail tipis/kecil < 4 piksel input hilang karena resolusi prototype;
- mask "terpotong lurus" kalau box-nya kekecilan (karena crop by box);
- objek bertumpuk dengan kelas sama bisa tertekan NMS.

Konversi: COCO → format label YOLO (polygon ternormalisasi). Objek dengan beberapa polygon (misal terpotong oleh objek lain) digabung dengan `merge_multi_segment` dari Ultralytics, sama seperti konverter resmi mereka.

In [ ]:
from ultralytics import YOLO
import yaml
try:
    from ultralytics.data.converter import merge_multi_segment
except Exception:
    merge_multi_segment = None

YOLO_DIR = f"{DATA_ROOT}/yolo"

def ann_to_polys(coco, a):
    seg = a['segmentation']
    if isinstance(seg, list):
        return [np.array(p, dtype=np.float32).reshape(-1, 2) for p in seg if len(p) >= 6]
    m = coco.annToMask(a)                                   # RLE -> kontur
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    return [c.reshape(-1, 2).astype(np.float32) for c in cnts if len(c) >= 3]

def write_yolo_split(split):
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(f"{CLEAN_DIR}/{split}.json")
    img_out, lbl_out = f"{YOLO_DIR}/images/{split}", f"{YOLO_DIR}/labels/{split}"
    os.makedirs(img_out, exist_ok=True); os.makedirs(lbl_out, exist_ok=True)
    n_multi = 0
    for img in coco.dataset['images']:
        dst = os.path.join(img_out, img['file_name'])
        if not os.path.lexists(dst): os.symlink(os.path.join(IMG_DIRS[split], img['file_name']), dst)
        W, H = img['width'], img['height']; lines = []
        for a in coco.loadAnns(coco.getAnnIds(imgIds=img['id'])):
            if a.get('iscrowd', 0): continue
            polys = ann_to_polys(coco, a)
            if not polys: continue
            if len(polys) > 1:
                n_multi += 1
                if merge_multi_segment is not None:
                    pts = np.concatenate(merge_multi_segment([p.reshape(-1).tolist() for p in polys]), 0)
                else:
                    pts = max(polys, key=cv2.contourArea)
            else:
                pts = polys[0]
            pts = (pts / np.array([W, H], dtype=np.float32)).clip(0, 1)
            lines.append(f"{CATID2IDX[a['category_id']]} " + " ".join(f"{v:.6f}" for v in pts.reshape(-1)))
        with open(os.path.join(lbl_out, os.path.splitext(img['file_name'])[0] + '.txt'), 'w') as f:
            f.write("\n".join(lines))
    print(f"{split}: {len(coco.imgs)} gambar, {n_multi} instance multi-polygon digabung")

for s in SPLITS:
    write_yolo_split(s)

def yolo_data_yaml():
    # daftar gambar subset train aktif (path absolut) + data.yaml untuk fraksi tsb
    sub = json.load(open(TRAIN_JSON))
    tag = f"f{int(round(TRAIN_FRACTION * 100))}"
    train_txt = f"{YOLO_DIR}/train_{tag}.txt"
    with open(train_txt, 'w') as f:
        f.write("\n".join(f"{YOLO_DIR}/images/train/{im['file_name']}" for im in sub['images']))
    path = f"{YOLO_DIR}/data_{tag}.yaml"
    yaml.safe_dump({'path': YOLO_DIR, 'train': train_txt, 'val': f"{YOLO_DIR}/images/valid",
                    'test': f"{YOLO_DIR}/images/test", 'names': dict(enumerate(CLASS_NAMES))}, open(path, 'w'))
    return path

In [ ]:
YOLO_WEIGHTS = 'yolo11s-seg.pt'      # pretrained COCO

@torch.no_grad()
def predict_yolo(weights, gt_json, img_dir):
    model = YOLO(weights)
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    res = []
    for img in coco.dataset['images']:
        r = model.predict(os.path.join(img_dir, img['file_name']), imgsz=IMG_SIZE, conf=0.001, iou=0.7,
                          max_det=100, retina_masks=True, verbose=False,
                          device=0 if DEVICE.type == 'cuda' else 'cpu')[0]
        if r.masks is None or len(r.boxes) == 0: continue
        H, W = img['height'], img['width']
        for m, c, s in zip(r.masks.data.cpu().numpy(), r.boxes.cls.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            if m.shape != (H, W): m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST)
            res.append({'image_id': img['id'], 'category_id': CAT_IDS[int(c)],
                        'segmentation': mask_to_rle(m > 0.5), 'score': float(s)})
    return res

def run_yolo():
    run = run_name('yolo11s-seg')
    if already_done(run): return
    seed_everything()
    with GPUTimer(run, 'YOLO11s-seg', note=f"frac={TRAIN_FRACTION}"):
        YOLO(YOLO_WEIGHTS).train(
            data=yolo_data_yaml(), imgsz=IMG_SIZE, epochs=MAX_EPOCHS, time=TIME_BUDGET_H,   # time= override epochs
            batch=16, seed=SEED, workers=2, amp=USE_AMP, device=0 if DEVICE.type == 'cuda' else 'cpu',
            project=f"{OUT_ROOT}/runs", name=run, exist_ok=True, plots=True)
    # log asli Ultralytics: {OUT_ROOT}/runs/{run}/results.csv
    preds = predict_yolo(f"{OUT_ROOT}/runs/{run}/weights/best.pt", TEST_JSON, IMG_DIRS['test'])
    save_run(run, 'YOLO11s-seg', 'one-stage CNN', 'test', coco_eval_segm(TEST_JSON, preds), preds)

## 5. Model B — Mask R-CNN R50-FPN v2 (two-stage, CNN)

**Cara kerja singkat**:
1. **Stage 1 (RPN)**: mengusulkan ~1000 region kandidat.
2. **Stage 2**: RoIAlign memotong fitur tiap region, lalu head box/kelas dan **mask head memprediksi grid 28×28** biner per kelas. Grid ini di-*resize* ke ukuran box di gambar asli.

Konsekuensi untuk analisis kegagalan:
- objek besar (kasur, lemari, dinding) diwakili hanya 28×28 sel, jadi tepinya kasar/"bocor" setelah di-upsample;
- kalau box salah (terlalu kecil), mask ikut terpotong;
- objek memanjang/tipis kehilangan detail.

Implementasi: `torchvision` (tanpa Detectron2, supaya instalasi di Colab tidak bermasalah). Input di-resize oleh model sehingga sisi terpanjang = `IMG_SIZE`.

In [ ]:
import torchvision
import torchvision.transforms.functional as TF
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

class MaskRCNNDataset(torch.utils.data.Dataset):
    def __init__(self, ann_json, img_dir, train=False):
        with contextlib.redirect_stdout(io.StringIO()):
            self.coco = COCO(ann_json)
        self.img_dir, self.train = img_dir, train
        self.ids = sorted(self.coco.imgs)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img_id = self.ids[i]; info = self.coco.imgs[img_id]
        img = TF.to_tensor(load_image(os.path.join(self.img_dir, info['file_name'])))
        masks, labels = load_instances(self.coco, img_id)
        masks = torch.from_numpy(masks)
        if self.train and random.random() < 0.5:                 # augmentasi: flip horizontal
            img, masks = img.flip(-1), masks.flip(-1)
        boxes = []
        for m in masks:                                          # box diturunkan dari mask -> konsisten
            ys, xs = torch.nonzero(m, as_tuple=True)
            boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
        target = {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
                  'labels': torch.from_numpy(labels) + 1,          # 0 = background di torchvision
                  'masks': masks}
        return img, target

def collate_tuple(batch): return tuple(zip(*batch))

def build_maskrcnn(num_classes=NUM_CLASSES, pretrained=True):
    w = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT if pretrained else None
    model = maskrcnn_resnet50_fpn_v2(weights=w, min_size=IMG_SIZE, max_size=IMG_SIZE)   # sisi terpanjang = IMG_SIZE
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, num_classes + 1)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels, 256, num_classes + 1)
    return model.to(DEVICE)

def mrcnn_step(model, batch):
    imgs, tgts = batch
    imgs = [i.to(DEVICE, non_blocking=True) for i in imgs]
    tgts = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in tgts]
    ld = model(imgs, tgts)
    loss = sum(ld.values())
    return loss, {'loss': loss.item(), **{k: v.item() for k, v in ld.items()}}

@torch.no_grad()
def predict_maskrcnn(model, gt_json, img_dir):
    model.eval(); model.roi_heads.score_thresh = 0.001; model.roi_heads.detections_per_img = 100
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    res = []
    for img in coco.dataset['images']:
        x = TF.to_tensor(load_image(os.path.join(img_dir, img['file_name']))).to(DEVICE)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model([x])[0]
        masks = (out['masks'][:, 0] > 0.5).cpu().numpy()
        for m, l, s in zip(masks, out['labels'].cpu().numpy(), out['scores'].float().cpu().numpy()):
            res.append({'image_id': img['id'], 'category_id': CAT_IDS[int(l) - 1],
                        'segmentation': mask_to_rle(m), 'score': float(s)})
    return res

In [ ]:
MRCNN_BS, MRCNN_LR = 4, 0.01      # LR SGD untuk batch 4 (skala linear dari resep torchvision)

def run_mrcnn():
    run = run_name('maskrcnn-r50fpnv2')
    if already_done(run): return
    seed_everything()
    model = build_maskrcnn()
    dl = torch.utils.data.DataLoader(MaskRCNNDataset(TRAIN_JSON, IMG_DIRS['train'], train=True), batch_size=MRCNN_BS,
                                     shuffle=True, num_workers=2, collate_fn=collate_tuple, pin_memory=True, drop_last=True)
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=MRCNN_LR, momentum=0.9, weight_decay=1e-4)
    val_fn = lambda m: coco_eval_segm(VAL_JSON, predict_maskrcnn(m, VAL_JSON, IMG_DIRS['valid']), verbose=False)
    with GPUTimer(run, 'Mask R-CNN R50-FPN v2', note=f"frac={TRAIN_FRACTION}"):
        best = train_loop(run, model, dl, mrcnn_step, opt, val_fn, clip=10.0)
    model.load_state_dict(torch.load(best, map_location=DEVICE))
    preds = predict_maskrcnn(model, TEST_JSON, IMG_DIRS['test'])
    save_run(run, 'Mask R-CNN R50-FPN v2', 'two-stage CNN', 'test', coco_eval_segm(TEST_JSON, preds), preds)

## 6. Model C — Mask2Former Swin-Tiny (transformer)

**Cara kerja singkat**:
1. Backbone **Swin Transformer** dan *pixel decoder* (multi-scale deformable attention) menghasilkan fitur per piksel di resolusi 1/4.
2. **Transformer decoder** dengan 100 *object query*. Tiap layer memakai **masked attention**: query hanya memperhatikan piksel di dalam mask prediksinya sendiri dari layer sebelumnya.
3. Tiap query mengeluarkan (kelas, mask). Training memakai **Hungarian matching** antara query dan GT, jadi tidak perlu NMS dan tidak ada box sama sekali.

Konsekuensi untuk analisis kegagalan:
- mask di 1/4 resolusi (mirip YOLO), tapi **tidak dibatasi box**, sehingga tidak ada efek "terpotong lurus", meski mask bisa "bocor" ke area dengan tekstur mirip;
- jumlah objek maksimum = 100 query;
- butuh data/epoch lebih banyak untuk konvergen. Ini relevan dengan variabel fraksi data.

**Preprocessing (ditulis manual)**: resize sisi terpanjang ke `IMG_SIZE`, pad ke persegi `IMG_SIZE×IMG_SIZE` (+`pixel_mask`), normalisasi ImageNet. Post-processing juga ditulis manual mengikuti paper: skor = prob kelas × rata-rata prob mask di area mask. Area padding dipotong sebelum mask dikembalikan ke resolusi asli.

Jika loss sering NaN dengan AMP: set `M2F_AMP = False` (lebih lambat ±2×) dan catat di logbook. Jika OOM: `M2F_BS = 2`.

In [ ]:
from transformers import Mask2FormerForUniversalSegmentation

M2F_CKPT = "facebook/mask2former-swin-tiny-coco-instance"
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def m2f_preprocess(img, S=IMG_SIZE):
    W, H = img.size; sc = S / max(H, W)
    nh, nw = max(1, round(H * sc)), max(1, round(W * sc))
    x = (TF.to_tensor(img.resize((nw, nh), Image.BILINEAR)) - IMNET_MEAN) / IMNET_STD
    pv = torch.zeros(3, S, S); pv[:, :nh, :nw] = x
    pm = torch.zeros(S, S, dtype=torch.long); pm[:nh, :nw] = 1
    return pv, pm, (nh, nw)

class M2FDataset(torch.utils.data.Dataset):
    def __init__(self, ann_json, img_dir, train=False):
        with contextlib.redirect_stdout(io.StringIO()):
            self.coco = COCO(ann_json)
        self.img_dir, self.train, self.ids = img_dir, train, sorted(self.coco.imgs)
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        img_id = self.ids[i]; info = self.coco.imgs[img_id]
        img = load_image(os.path.join(self.img_dir, info['file_name']))
        masks, labels = load_instances(self.coco, img_id)
        if self.train and random.random() < 0.5:
            img = ImageOps.mirror(img); masks = masks[:, :, ::-1]
        pv, pm, (nh, nw) = m2f_preprocess(img)
        ml = torch.zeros(len(masks), IMG_SIZE, IMG_SIZE)
        for k, m in enumerate(masks):
            ml[k, :nh, :nw] = torch.from_numpy(cv2.resize(np.ascontiguousarray(m), (nw, nh), interpolation=cv2.INTER_NEAREST)).float()
        keep = ml.flatten(1).sum(1) > 0                          # instance yang hilang setelah resize dibuang
        return {'pixel_values': pv, 'pixel_mask': pm, 'mask_labels': ml[keep],
                'class_labels': torch.from_numpy(labels)[keep]}

def m2f_collate(batch):
    return {'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
            'pixel_mask': torch.stack([b['pixel_mask'] for b in batch]),
            'mask_labels': [b['mask_labels'] for b in batch],
            'class_labels': [b['class_labels'] for b in batch]}

def build_m2f(pretrained=True):
    id2label = dict(enumerate(CLASS_NAMES))
    if pretrained:
        model = Mask2FormerForUniversalSegmentation.from_pretrained(
            M2F_CKPT, id2label=id2label, label2id={v: k for k, v in id2label.items()}, ignore_mismatched_sizes=True)
    else:
        from transformers import Mask2FormerConfig
        cfg = Mask2FormerConfig.from_pretrained(M2F_CKPT, id2label=id2label, label2id={v: k for k, v in id2label.items()})
        model = Mask2FormerForUniversalSegmentation(cfg)
    # bobot kelas "no object" pada loss: ukurannya berubah (81 -> K+1), set ulang secara eksplisit
    ew = torch.ones(NUM_CLASSES + 1); ew[-1] = model.config.no_object_weight
    model.criterion.empty_weight.copy_(ew)
    return model.to(DEVICE)

def m2f_step(model, batch):
    out = model(pixel_values=batch['pixel_values'].to(DEVICE, non_blocking=True),
                pixel_mask=batch['pixel_mask'].to(DEVICE, non_blocking=True),
                mask_labels=[m.to(DEVICE) for m in batch['mask_labels']],
                class_labels=[c.to(DEVICE) for c in batch['class_labels']])
    return out.loss, {'loss': out.loss.item()}

@torch.no_grad()
def m2f_postprocess(cls_logits, mask_logits, nh, nw, H, W, S=IMG_SIZE, topk=100, chunk=10):
    # cls_logits (Q,K+1), mask_logits (Q,h,w) -> list (mask bool HxW, label, score)
    # Diproses per `chunk` query: 100 mask float di resolusi asli foto besar (~20 MP) butuh ~8 GB -> OOM di T4.
    # Hasilnya identik dengan memproses semua query sekaligus.
    K = cls_logits.shape[-1] - 1
    probs = cls_logits.float().softmax(-1)[:, :-1]              # buang kelas "no object"
    scores, idx = probs.flatten().topk(min(topk, probs.numel()))
    labels, qidx = idx % K, idx // K
    masks, mscores = [], []
    for c in range(0, len(qidx), chunk):
        m = F.interpolate(mask_logits[qidx[c:c + chunk]][None].float(), size=(S, S), mode='bilinear', align_corners=False)[0]
        m = F.interpolate(m[None, :, :nh, :nw], size=(H, W), mode='bilinear', align_corners=False)[0]   # buang padding
        prob = m.sigmoid(); binm = prob > 0.5
        mscores.append((prob * binm).flatten(1).sum(1) / (binm.flatten(1).sum(1) + 1e-6))
        masks.append(binm.cpu().numpy()); del m, prob, binm
    mscore = torch.cat(mscores) if mscores else scores.new_zeros(0)
    binm = np.concatenate(masks) if masks else np.zeros((0, H, W), bool)
    return binm, labels.cpu().numpy(), (scores * mscore).cpu().numpy()

@torch.no_grad()
def predict_m2f(model, gt_json, img_dir):
    model.eval()
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    res = []
    for img in coco.dataset['images']:
        pil = load_image(os.path.join(img_dir, img['file_name']))
        pv, pm, (nh, nw) = m2f_preprocess(pil)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(pixel_values=pv[None].to(DEVICE), pixel_mask=pm[None].to(DEVICE))
        masks, labels, scores = m2f_postprocess(out.class_queries_logits[0], out.masks_queries_logits[0],
                                                nh, nw, pil.size[1], pil.size[0])
        for m, l, s in zip(masks, labels, scores):
            if m.any():
                res.append({'image_id': img['id'], 'category_id': CAT_IDS[int(l)],
                            'segmentation': mask_to_rle(m), 'score': float(s)})
    return res

In [ ]:
M2F_BS, M2F_LR, M2F_AMP = 4, 1e-4, USE_AMP     # resep paper: AdamW 1e-4, backbone x0.1, wd 0.05, clip 0.01

def run_m2f():
    run = run_name('mask2former-swinT')
    if already_done(run): return
    seed_everything()
    model = build_m2f()
    dl = torch.utils.data.DataLoader(M2FDataset(TRAIN_JSON, IMG_DIRS['train'], train=True), batch_size=M2F_BS,
                                     shuffle=True, num_workers=2, collate_fn=m2f_collate, pin_memory=True, drop_last=True)
    backbone = [p for n, p in model.named_parameters() if 'pixel_level_module.encoder' in n]
    others   = [p for n, p in model.named_parameters() if 'pixel_level_module.encoder' not in n]
    opt = torch.optim.AdamW([{'params': backbone, 'lr': M2F_LR * 0.1}, {'params': others, 'lr': M2F_LR}], weight_decay=0.05)
    val_fn = lambda m: coco_eval_segm(VAL_JSON, predict_m2f(m, VAL_JSON, IMG_DIRS['valid']), verbose=False)
    with GPUTimer(run, 'Mask2Former Swin-T', note=f"frac={TRAIN_FRACTION}, amp={M2F_AMP}"):
        best = train_loop(run, model, dl, m2f_step, opt, val_fn, amp=M2F_AMP, clip=0.01)
    model.load_state_dict(torch.load(best, map_location=DEVICE))
    preds = predict_m2f(model, TEST_JSON, IMG_DIRS['test'])
    save_run(run, 'Mask2Former Swin-T', 'transformer', 'test', coco_eval_segm(TEST_JSON, preds), preds)

## 10. Input evaluasi
Mencari otomatis file input di `/kaggle/input`. Hasil training (results, GPU log) disalin ke output notebook ini supaya rekap lengkap di satu tempat.

In [ ]:
import re, shutil
INPUT_ROOT = '/kaggle/input' if IN_KAGGLE else os.path.abspath('eval_input')   # Colab/lokal: taruh semua input di folder ini

def find_one(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"{what} tidak ditemukan ({pattern}). Cek Add Input di panel kanan.")
    if len(hits) > 1: print(f"  ({what}: {len(hits)} kandidat, dipakai yang pertama)")
    return hits[0]

CUSTOM_JSON    = find_one(f"{INPUT_ROOT}/**/test_custom_clean.json", 'test_custom_clean.json')
CONDITIONS_CSV = find_one(f"{INPUT_ROOT}/**/conditions.csv", 'conditions.csv')
TRAIN_OUT      = os.path.dirname(os.path.dirname(
                    find_one(f"{INPUT_ROOT}/**/results/*__test.json", 'output training (results/*__test.json)')))
# run dicari dari BOBOT (runs/), bukan hanya dari results/: run yang training-nya selesai tapi evaluasi
# test aslinya gagal di notebook training tetap ikut, dan hasil test-nya dihitung ulang di Bagian 11
with_results = {os.path.basename(p).split('__')[0] for p in glob.glob(f"{TRAIN_OUT}/results/*__test.json")}
with_weights = {os.path.basename(os.path.dirname(p.replace('/weights', ''))) for p in
                glob.glob(f"{TRAIN_OUT}/runs/*/best.pt") + glob.glob(f"{TRAIN_OUT}/runs/*/last.pt") +
                glob.glob(f"{TRAIN_OUT}/runs/*/weights/best.pt") + glob.glob(f"{TRAIN_OUT}/runs/*/weights/last.pt")}
RUNS = sorted(with_results | with_weights)
print('anotasi custom :', CUSTOM_JSON)
print('kondisi        :', CONDITIONS_CSV)
print('output training:', TRAIN_OUT)
print(f'{len(RUNS)} run:', RUNS)
print('⚠️  ada bobot tapi TANPA hasil test asli (dihitung ulang di Bagian 11):', sorted(with_weights - with_results) or '-')
print('⚠️  ada hasil tapi TANPA bobot (tidak bisa dievaluasi di test custom):', sorted(with_results - with_weights) or '-')
if os.path.exists(f"{TRAIN_OUT}/LOGBOOK.md"):
    print('\n--- LOGBOOK training ---\n' + open(f"{TRAIN_OUT}/LOGBOOK.md").read())

for p in glob.glob(f"{TRAIN_OUT}/results/*__test.json"):        # hasil test asli -> rekap
    shutil.copy(p, f"{OUT_ROOT}/results/")
if os.path.exists(f"{TRAIN_OUT}/gpu_time_log.csv") and not os.path.exists(GPU_LOG):
    shutil.copy(f"{TRAIN_OUT}/gpu_time_log.csv", GPU_LOG)    # gpu_minutes(run) tetap = waktu training

### 10.1 Siapkan test custom
1. Resize foto + skala anotasi ke resolusi test asli.
2. `clean_split` (fungsi yang sama dengan Bagian 2.2) menyamakan id kelas lewat **nama**. Kelas yang tidak ada di train ikut dibuang dan dilaporkan.
3. `test_sub` = test asli, tetapi hanya anotasi dari kelas yang muncul di test custom.

In [ ]:
cj = json.load(open(CUSTOM_JSON))
first = cj['images'][0]['file_name']
CUSTOM_SRC_DIR = os.path.dirname(find_one(f"{INPUT_ROOT}/**/{glob.escape(first)}", 'gambar test custom'))

test_imgs   = json.load(open(TEST_JSON))['images']
TARGET_LONG = int(np.median([max(i['width'], i['height']) for i in test_imgs]))
CUSTOM_IMG_DIR = f"{DATA_ROOT}/custom_imgs"; os.makedirs(CUSTOM_IMG_DIR, exist_ok=True)

scale = {}
for im in cj['images']:
    s = min(1.0, TARGET_LONG / max(im['width'], im['height']))
    pil = load_image(os.path.join(CUSTOM_SRC_DIR, im['file_name']))
    if s < 1.0:
        pil = pil.resize((round(im['width'] * s), round(im['height'] * s)), Image.LANCZOS)
    pil.save(os.path.join(CUSTOM_IMG_DIR, im['file_name']), quality=95)
    im['width'], im['height'] = pil.size
    scale[im['id']] = s
for a in cj['annotations']:
    s = scale[a['image_id']]
    a['segmentation'] = [[v * s for v in p] for p in a['segmentation']]
    a['bbox'] = [v * s for v in a['bbox']]
    a['area'] = a['area'] * s * s

out, dropped = clean_split(cj)
CUSTOM_GT = f"{CLEAN_DIR}/test_custom.json"; json.dump(out, open(CUSTOM_GT, 'w'))
IMG_DIRS['test_custom'] = CUSTOM_IMG_DIR
CUSTOM_CATS = sorted({a['category_id'] for a in out['annotations']})
CUSTOM_CLASSES = [cats_by_id[c]['name'] for c in CUSTOM_CATS]
print(f"test_custom: {len(out['images'])} gambar, {len(out['annotations'])} instance, {len(CUSTOM_CATS)} kelas, "
      f"resize sisi terpanjang -> {TARGET_LONG}px, dibuang: {dropped or '-'}")

tj = json.load(open(TEST_JSON))
TEST_SUB_JSON = f"{CLEAN_DIR}/test_sub.json"
json.dump(dict(tj, annotations=[a for a in tj['annotations'] if a['category_id'] in CUSTOM_CATS]), open(TEST_SUB_JSON, 'w'))
n_sub = defaultdict(int)
for a in tj['annotations']:
    if a['category_id'] in CUSTOM_CATS: n_sub[a['category_id']] += 1
print(f"test_sub   : {sum(n_sub.values())} instance dari {len(n_sub)} kelas yang sama")
print("kelas custom yang TIDAK ada di test asli (tidak ikut dibandingkan per kelas):",
      [cats_by_id[c]['name'] for c in CUSTOM_CATS if c not in n_sub] or '-')

# kondisi per foto (conditions.csv memakai nama file asli sebelum Roboflow menambah '_jpg.rf.<hash>')
cond = pd.read_csv(CONDITIONS_CSV)
stem = lambda fn: re.sub(r'_jpg\.rf\..*$', '', fn) + '.jpg'
img_df = pd.DataFrame([{'image_id': im['id'], 'file': stem(im['file_name']), 'file_name': im['file_name']}
                       for im in out['images']]).merge(cond, on='file', how='left')
missing = img_df[img_df.kondisi_utama.isna()].file.tolist()
print("foto tanpa kondisi di conditions.csv:", missing or '-')
display(img_df.kondisi_utama.value_counts().rename('jumlah foto').to_frame())
show_gt('test_custom', n=4)

## 11. Inferensi 9 model di test custom
- `test_custom`: inferensi ulang memakai bobot `best` tiap run.
- `test_sub`: **tidak** perlu inferensi ulang. Prediksi test asli yang tersimpan dari notebook training dinilai ulang terhadap `test_sub.json`.
- Run yang hasilnya sudah ada dilewati. Waktu inferensi tercatat di `gpu_time_log.csv` dengan nama `<run>__eval_custom`.

In [ ]:
MODEL_META = {'yolo': ('YOLO11s-seg', 'one-stage CNN'),
              'maskrcnn': ('Mask R-CNN R50-FPN v2', 'two-stage CNN'),
              'mask2former': ('Mask2Former Swin-T', 'transformer')}

def meta(run):
    return next(v for k, v in MODEL_META.items() if run.startswith(k))

def load_and_predict(run, gt_json, img_dir):
    w = f"{TRAIN_OUT}/runs/{run}"
    if run.startswith('yolo'):
        wt = f"{w}/weights/best.pt" if os.path.exists(f"{w}/weights/best.pt") else f"{w}/weights/last.pt"
        return predict_yolo(wt, gt_json, img_dir)
    wt = f"{w}/best.pt" if os.path.exists(f"{w}/best.pt") else f"{w}/last.pt"
    model = build_maskrcnn(pretrained=False) if run.startswith('maskrcnn') else build_m2f(pretrained=False)
    model.load_state_dict(torch.load(wt, map_location=DEVICE))
    fn = predict_maskrcnn if run.startswith('maskrcnn') else predict_m2f
    try:
        return fn(model, gt_json, img_dir)
    finally:
        del model

for run in RUNS:
    TRAIN_FRACTION = int(re.search(r'_f(\d+)_', run).group(1)) / 100     # dibaca save_run
    name, fam = meta(run)
    try:
        if not os.path.exists(f"{OUT_ROOT}/results/{run}__test_custom.json"):
            with GPUTimer(f"{run}__eval_custom", name, note='inferensi test_custom'):
                preds = load_and_predict(run, CUSTOM_GT, CUSTOM_IMG_DIR)
            save_run(run, name, fam, 'test_custom', coco_eval_segm(CUSTOM_GT, preds), preds)
        need_test = not os.path.exists(f"{OUT_ROOT}/results/{run}__test.json")
        if need_test or not os.path.exists(f"{OUT_ROOT}/results/{run}__test_sub.json"):
            tp = f"{TRAIN_OUT}/preds/{run}__test.json"
            if os.path.exists(tp):
                preds_t = json.load(open(tp))
            else:
                print(f"  {run}: prediksi test asli tidak ada di output training -> inferensi ulang")
                with GPUTimer(f"{run}__eval_test", name, note='inferensi ulang test asli'):
                    preds_t = load_and_predict(run, TEST_JSON, IMG_DIRS['test'])
            if need_test:
                save_run(run, name, fam, 'test', coco_eval_segm(TEST_JSON, preds_t), preds_t,
                         extra={'note': 'dihitung ulang di notebook eval (evaluasi di notebook training gagal)'})
            save_run(run, name, fam, 'test_sub', coco_eval_segm(TEST_SUB_JSON, preds_t), [],
                     extra={'preds_source': f"{run}__test.json (dinilai ulang dgn kelas test custom)"})
    except Exception as e:
        traceback.print_exc(); log_note(f"GAGAL eval {run}: {type(e).__name__}: {str(e)[:200]}")
    finally:
        gc.collect(); torch.cuda.empty_cache()

## 12. Domain shift: test asli vs test custom
- `test` = test asli, semua kelas (angka dari notebook training, hanya untuk referensi).
- `test_sub` = test asli, kelas sama dengan test custom → **pembanding yang adil**.
- `drop` = `test_custom − test_sub`; `drop_%` = penurunan relatif.

In [ ]:
recs = [json.load(open(p)) for p in sorted(glob.glob(f"{OUT_ROOT}/results/*.json"))]
res_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'per_class'} for r in recs])
SPL = ['test', 'test_sub', 'test_custom']
ds = res_df.pivot_table(index=['model', 'train_fraction'], columns='eval_split', values=['AP', 'AP50'])
assert ('AP', 'test_custom') in ds and ('AP', 'test_sub') in ds, "belum ada hasil test_custom/test_sub — cek error & LOGBOOK di Bagian 11"
for m in ['AP', 'AP50']:
    ds[(m, 'drop')]   = ds[(m, 'test_custom')] - ds[(m, 'test_sub')]
    ds[(m, 'drop_%')] = 100 * ds[(m, 'drop')] / ds[(m, 'test_sub')]
ds = ds[[(m, s) for m in ['AP', 'AP50'] for s in SPL + ['drop', 'drop_%'] if (m, s) in ds.columns]]
display(ds.round(4))
os.makedirs(f"{OUT_ROOT}/figures", exist_ok=True)
ds.to_csv(f"{OUT_ROOT}/domain_shift.csv")
res_df.drop(columns=[c for c in ['preds_source'] if c in res_df]).to_csv(f"{OUT_ROOT}/summary_metrics.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for (mname, g), col in zip(res_df.groupby('model'), plt.cm.tab10.colors):
    for split, ls in [('test_sub', '-'), ('test_custom', '--')]:
        h = g[g.eval_split == split].sort_values('train_fraction')
        for a, met in zip(ax, ['AP', 'AP50']):
            a.plot(h.train_fraction * 100, h[met], 'o' + ls, color=col, label=f"{mname} · {split}")
for a, t in zip(ax, ['mask AP@[.5:.95]', 'mask AP50']):
    a.set_xlabel('% data latih'); a.set_ylabel(t); a.set_xticks([25, 50, 100]); a.grid(alpha=.3)
ax[1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.suptitle('Garis penuh = test asli (kelas sama), putus-putus = test custom'); plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/figures/domain_shift_ap.png", dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# AP per kelas (fraksi 100%): kelas mana yang paling turun di foto HP
n_cust = pd.Series([cats_by_id[a['category_id']]['name'] for a in json.load(open(CUSTOM_GT))['annotations']]).value_counts()
rows = []
for r in recs:
    if r['eval_split'] in ('test_sub', 'test_custom') and abs(r['train_fraction'] - 1.0) < 1e-6:
        for c in CUSTOM_CLASSES:
            rows.append(dict(model=r['model'], split=r['eval_split'], cls=c, AP=r['per_class'].get(c, np.nan)))
pcs = pd.DataFrame(rows).pivot_table(index='cls', columns=['model', 'split'], values='AP')
for m in pcs.columns.get_level_values(0).unique():
    if (m, 'test_sub') in pcs and (m, 'test_custom') in pcs:
        pcs[(m, 'drop')] = pcs[(m, 'test_custom')] - pcs[(m, 'test_sub')]
pcs = pcs.sort_index(axis=1)
pcs.insert(0, 'n_custom', n_cust.reindex(pcs.index).fillna(0).astype(int))
display(pcs.sort_values('n_custom', ascending=False).round(3))
pcs.to_csv(f"{OUT_ROOT}/per_class_domain_shift_f100.csv")
print("Catatan: kelas dengan n_custom kecil (< 5) angkanya sangat tidak stabil; jangan disimpulkan sendirian.")

### 12.1 Seberapa yakin? Bootstrap confidence interval
Test custom hanya 41 foto, jadi selisih AP bisa muncul karena kebetulan pemilihan foto. **Bootstrap**: ambil ulang foto secara acak *dengan pengembalian* (jumlahnya sama), hitung AP, ulangi `BOOT_B` kali. Rentang persentil 2.5–97.5% = **CI 95%**.

- CI dihitung terpisah untuk `test_sub` dan `test_custom` (dua set foto independen). CI `drop` dihitung dari selisih sampel bootstrap keduanya.
- Dipakai **CI bootstrap "basic"** (reverse percentile), bukan persentil biasa. Alasannya, AP pada sampel kecil **bias ke atas**: amplop precision mengambil nilai maksimum, dan sampel bootstrap praktis hanya berisi sekitar 63% foto unik. Kolom `bias_boot_custom` menunjukkan seberapa besar bias itu. Sebutkan di laporan sebagai keterbatasan metode.
- **Kalau CI `drop` memuat 0 → tidak ada bukti penurunan yang signifikan.** Kalimat di laporan harus mengikuti ini.
- Implementasi: `COCOeval.evaluate()` dijalankan **sekali**, lalu langkah *accumulate* (precision 101 titik recall, 10 ambang IoU, maxDet 100) ditulis ulang supaya bisa dihitung untuk subset foto dengan duplikat. Self-check di bawah memastikan hasilnya sama dengan `E.stats` pycocotools.

In [ ]:
BOOT_B, BOOT_SEED = 1000, 0
REC_THRS = np.linspace(0, 1, 101)

def coco_evalimgs(gt_json, preds, iou_thrs=None):
    # COCOeval.evaluate() saja (area 'all'), untuk accumulate manual
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(gt_json); dt = gt.loadRes(copy.deepcopy(preds))
        E = COCOeval(gt, dt, 'segm')
        if iou_thrs is not None: E.params.iouThrs = np.array(iou_thrs, dtype=float)
        E.params.areaRng, E.params.areaRngLbl = [E.params.areaRng[0]], ['all']
        E.evaluate()
    return E

def per_image_records(E):
    # {cat: {img: (scores, matched TxD, ignored TxD, n_gt_valid)}}
    I, recs = len(E.params.imgIds), {}
    for k, cid in enumerate(E.params.catIds):
        per = {}
        for i, img in enumerate(E.params.imgIds):
            e = E.evalImgs[k * I + i]
            if e is None: continue
            per[img] = (np.asarray(e['dtScores'], float), e['dtMatches'] != 0, e['dtIgnore'].astype(bool),
                        int((np.asarray(e['gtIgnore']) == 0).sum()))
        recs[cid] = per
    return recs

def ap_from_records(recs, img_list, T=10):
    # sama dengan COCOeval.accumulate + summarize (area all, maxDet 100); img_list boleh berisi duplikat
    aps = []
    for per in recs.values():
        parts = [per[i] for i in img_list if i in per]
        npig = sum(p[3] for p in parts)
        if npig == 0: continue
        sc = np.concatenate([p[0] for p in parts]) if parts else np.zeros(0)
        if sc.size == 0: aps.append(np.zeros(T)); continue
        order = np.argsort(-sc, kind='mergesort')
        dm = np.concatenate([p[1] for p in parts], 1)[:, order]
        di = np.concatenate([p[2] for p in parts], 1)[:, order]
        tp = np.cumsum(dm & ~di, 1).astype(float); fp = np.cumsum(~dm & ~di, 1).astype(float)
        ap_t = np.zeros(T)
        for t in range(T):
            rc = tp[t] / npig
            pr = np.maximum.accumulate((tp[t] / np.maximum(tp[t] + fp[t], np.spacing(1)))[::-1])[::-1]
            idx = np.searchsorted(rc, REC_THRS, side='left')
            ap_t[t] = np.where(idx < len(pr), pr[np.minimum(idx, len(pr) - 1)], 0.0).mean()
        aps.append(ap_t)
    if not aps: return np.nan, np.nan
    A = np.array(aps)
    return float(A.mean()), float(A[:, 0].mean())          # AP@[.5:.95], AP50

def bootstrap_ap(recs, img_ids, B=BOOT_B, seed=BOOT_SEED):
    rng, ids = np.random.default_rng(seed), np.asarray(img_ids)
    return np.array([ap_from_records(recs, ids[rng.integers(0, len(ids), len(ids))]) for _ in range(B)])

def test_preds_of(run):
    for p in [f"{TRAIN_OUT}/preds/{run}__test.json", f"{OUT_ROOT}/preds/{run}__test.json"]:
        if os.path.exists(p): return json.load(open(p))
    raise FileNotFoundError(f"prediksi test asli {run} tidak ada")

ci = lambda x: (np.nanpercentile(x, 2.5), np.nanpercentile(x, 97.5))        # persentil (dipakai untuk recall)
def ci_basic(full, x):
    # CI "basic"/reverse-percentile: koreksi bias bootstrap. AP pada sampel kecil bias ke ATAS (amplop precision
    # mengambil maksimum), jadi interval persentil biasa akan bergeser ke atas.
    lo, hi = ci(x); return 2 * full - hi, 2 * full - lo
rows, t0 = [], time.time()
for run in RUNS:
    try:
        name, _ = meta(run); frac = int(re.search(r'_f(\d+)_', run).group(1))
        boots = {}
        for split, gtj, preds in [('test_sub', TEST_SUB_JSON, test_preds_of(run)),
                                  ('test_custom', CUSTOM_GT, json.load(open(f"{OUT_ROOT}/preds/{run}__test_custom.json")))]:
            E = coco_evalimgs(gtj, preds); recs = per_image_records(E)
            full = ap_from_records(recs, E.params.imgIds)
            with contextlib.redirect_stdout(io.StringIO()):
                E.params.areaRng, E.params.areaRngLbl = [[0, 1e10]], ['all']; E.accumulate(); E.summarize()
            assert np.allclose(full, (E.stats[0], E.stats[1]), atol=1e-6), f"self-check gagal {run} {split}: {full} vs {E.stats[:2]}"
            boots[split] = (full, bootstrap_ap(recs, E.params.imgIds, seed=BOOT_SEED + frac))
        for j, met in enumerate(['AP', 'AP50']):
            (fs, bs), (fc, bc) = boots['test_sub'], boots['test_custom']
            drop = fc[j] - fs[j]; lo, hi = ci_basic(drop, bc[:, j] - bs[:, j])
            (sl, sh), (cl, ch) = ci_basic(fs[j], bs[:, j]), ci_basic(fc[j], bc[:, j])
            rows.append(dict(model=name, train_fraction=frac / 100, metric=met,
                             test_sub=fs[j], test_sub_lo=sl, test_sub_hi=sh,
                             test_custom=fc[j], test_custom_lo=cl, test_custom_hi=ch,
                             drop=drop, drop_lo=lo, drop_hi=hi,
                             signifikan='ya (turun)' if hi < 0 else ('ya (naik)' if lo > 0 else 'tidak'),
                             bias_boot_custom=np.nanmean(bc[:, j]) - fc[j]))
        print(f"✓ {run}  ({(time.time() - t0) / 60:.1f} menit)")
    except Exception as e:
        traceback.print_exc(); log_note(f"GAGAL bootstrap {run}: {type(e).__name__}: {str(e)[:200]}")
boot_df = pd.DataFrame(rows).sort_values(['metric', 'model', 'train_fraction'])
display(boot_df.round(4))
boot_df.to_csv(f"{OUT_ROOT}/domain_shift_bootstrap_ci.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, met in zip(ax, ['AP', 'AP50']):
    g = boot_df[boot_df.metric == met].reset_index(drop=True)
    x = np.arange(len(g))
    for off, sp, mk in [(-0.15, 'test_sub', 'o'), (0.15, 'test_custom', 's')]:
        a.errorbar(x + off, g[sp], yerr=[g[sp] - g[f'{sp}_lo'], g[f'{sp}_hi'] - g[sp]], fmt=mk, capsize=3, label=sp)
    a.set_xticks(x); a.set_xticklabels([f"{m.split()[0]}\n{f:.0%}" for m, f in zip(g.model, g.train_fraction)], fontsize=8)
    a.set_ylabel(f'mask {met}'); a.grid(alpha=.3); a.legend()
plt.suptitle(f'Domain shift dengan CI 95% bootstrap (B={BOOT_B})'); plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/figures/domain_shift_ci.png", dpi=150); plt.show()

## 13. AP per kondisi sulit
Foto custom dikelompokkan menurut `kondisi_utama` di `conditions.csv`, lalu tiap kelompok dievaluasi terpisah (fraksi 100%).

**Hati-hati menafsirkan**: tiap kelompok hanya 1–9 foto, jadi selisih kecil antar kondisi bisa kebetulan. Pembanding paling stabil adalah **`SEMUA SULIT` vs `normal`**. Pola yang **konsisten di ketiga model** lebih bisa dipercaya daripada angka satu model.

⚠️ **AP per kelompok tidak bisa dibandingkan antar kelompok secara langsung.** AP dirata-rata *per kelas*, dan tiap kelompok berisi kelas yang berbeda, sehingga kelompok yang kebetulan berisi kelas mudah mendapat AP tinggi. Selain itu, di subset kecil, prediksi salah dari foto lain tidak ikut bersaing di ranking skor. Itu sebabnya `SEMUA FOTO` bisa lebih rendah daripada hampir semua kelompoknya. Untuk membandingkan kondisi, pakai **recall per instance di 13.1**. Tabel ini disimpan hanya sebagai pelengkap.

In [ ]:
def eval_subset(gt_json, preds, img_ids, tag):
    keep = set(img_ids); js = json.load(open(gt_json))
    sub = dict(js, images=[i for i in js['images'] if i['id'] in keep],
               annotations=[a for a in js['annotations'] if a['image_id'] in keep])
    p = f"{CLEAN_DIR}/_subset_{tag}.json"; json.dump(sub, open(p, 'w'))
    return coco_eval_segm(p, [d for d in preds if d['image_id'] in keep], verbose=False)

groups = {k: g.image_id.tolist() for k, g in img_df.dropna(subset=['kondisi_utama']).groupby('kondisi_utama')}
groups['SEMUA SULIT'] = img_df[img_df.kondisi_utama.notna() & (img_df.kondisi_utama != 'normal')].image_id.tolist()
groups['SEMUA FOTO'] = img_df.image_id.tolist()

F100 = [r for r in RUNS if '_f100_' in r]
rows = []
for run in F100:
    preds = json.load(open(f"{OUT_ROOT}/preds/{run}__test_custom.json"))
    for k, ids in groups.items():
        m = eval_subset(CUSTOM_GT, preds, ids, 'cond')
        rows.append(dict(model=meta(run)[0], kondisi=k, n_foto=len(ids), AP=m['AP'], AP50=m['AP50']))
cond_df = pd.DataFrame(rows)
cond_tab = cond_df.pivot_table(index=['kondisi', 'n_foto'], columns='model', values=['AP50', 'AP'])
cond_tab = cond_tab.sort_values(cond_tab.columns[0])
display(cond_tab.round(3))
cond_tab.to_csv(f"{OUT_ROOT}/ap_per_kondisi_f100.csv")

hm = cond_df.pivot_table(index='kondisi', columns='model', values='AP50')
hm = hm.loc[hm.mean(1).sort_values().index]
plt.figure(figsize=(7, 0.4 * len(hm) + 1.5))
plt.imshow(hm.values, aspect='auto', cmap='magma', vmin=0)
plt.yticks(range(len(hm)), [f"{k} (n={len(groups[k])})" for k in hm.index]); plt.xticks(range(hm.shape[1]), hm.columns, rotation=20)
for i in range(hm.shape[0]):
    for j in range(hm.shape[1]):
        plt.text(j, i, f"{hm.values[i, j]:.2f}", ha='center', va='center', color='w', fontsize=8)
plt.colorbar(label='mask AP50'); plt.title('AP50 per kondisi (fraksi 100%)'); plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/figures/ap_per_kondisi.png", dpi=150); plt.show()

### 13.1 Recall per instance per kondisi (metrik utama untuk analisis kondisi)
**Recall** = dari semua objek GT di kelompok foto itu, berapa persen yang **ditemukan** oleh model: ada prediksi dengan **kelas benar**, **mask IoU ≥ 0.5**, dan **skor ≥ `REC_SCORE_THR`**. Pencocokannya memakai aturan COCO (greedy per kelas, satu prediksi untuk satu objek).

Berbeda dengan AP per kelompok, recall dihitung **per objek**, jadi tidak terdistorsi oleh rata-rata per kelas. Baris `TEST ASLI (test_sub)` menjadi garis acuan, yaitu performa model di domain training.

CI 95% dihitung dengan bootstrap **per foto**, karena objek dalam satu foto tidak saling independen. Kelompok dengan sedikit foto akan punya CI lebar. **Jangan menyimpulkan perbedaan antar kondisi kalau CI-nya saling tumpang tindih.**

Keterbatasan: recall tidak menghukum prediksi salah (false positive). Prediksi salah tetap terlihat di AP (Bagian 12) dan di visualisasi (Bagian 14).

In [ ]:
REC_SCORE_THR = 0.3

def per_image_hits(gt_json, preds, cat_ids):
    # -> {img: (objek ditemukan, total objek)} pada IoU 0.5, skor >= REC_SCORE_THR
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(gt_json)
    tot = defaultdict(int)
    for a in gt.dataset['annotations']:
        if not a.get('iscrowd', 0) and a['category_id'] in cat_ids: tot[a['image_id']] += 1
    p = [d for d in preds if d['score'] >= REC_SCORE_THR and d['category_id'] in cat_ids]
    hit = defaultdict(int)
    if p:
        E = coco_evalimgs(gt_json, p, iou_thrs=[0.5]); I = len(E.params.imgIds)
        for k, cid in enumerate(E.params.catIds):
            for i, img in enumerate(E.params.imgIds):
                e = E.evalImgs[k * I + i]
                if e is None: continue
                ok = np.asarray(e['gtIgnore']) == 0
                hit[img] += int(((e['gtMatches'][0] != 0) & ok).sum())
    return {i: (hit[i], tot[i]) for i in gt.getImgIds()}

def recall_ci(hits, img_ids, B=BOOT_B, seed=BOOT_SEED):
    h = np.array([hits[i][0] for i in img_ids], float); t = np.array([hits[i][1] for i in img_ids], float)
    rng = np.random.default_rng(seed); n = len(img_ids)
    bs = []
    for _ in range(B):
        s = rng.integers(0, n, n)
        if t[s].sum() > 0: bs.append(h[s].sum() / t[s].sum())
    lo, hi = ci(bs) if n >= 3 else (np.nan, np.nan)     # < 3 foto: CI tidak bermakna (1 foto -> lebar nol, menyesatkan)
    return h.sum() / max(t.sum(), 1), lo, hi, int(t.sum())

rows = []
for run in F100:
    name, _ = meta(run)
    hc = per_image_hits(CUSTOM_GT, json.load(open(f"{OUT_ROOT}/preds/{run}__test_custom.json")), set(CUSTOM_CATS))
    hs = per_image_hits(TEST_SUB_JSON, test_preds_of(run), set(CUSTOM_CATS))
    r, lo, hi, n = recall_ci(hs, sorted(hs))
    rows.append(dict(model=name, kondisi='TEST ASLI (test_sub)', n_foto=len(hs), n_objek=n, recall=r, lo=lo, hi=hi))
    for k, ids in groups.items():
        r, lo, hi, n = recall_ci(hc, ids)
        rows.append(dict(model=name, kondisi=k, n_foto=len(ids), n_objek=n, recall=r, lo=lo, hi=hi))
rec_df = pd.DataFrame(rows)
rt = rec_df.pivot_table(index=['kondisi', 'n_foto', 'n_objek'], columns='model', values=['recall', 'lo', 'hi'])
display(rt[[(v, m) for m in rec_df.model.unique() for v in ['recall', 'lo', 'hi']]].round(3))
rec_df.to_csv(f"{OUT_ROOT}/recall_per_kondisi_f100.csv", index=False)

order = ['TEST ASLI (test_sub)', 'SEMUA FOTO', 'SEMUA SULIT', 'normal'] + \
        sorted(k for k in groups if k not in ('SEMUA FOTO', 'SEMUA SULIT', 'normal'))
models = rec_df.model.unique()
fig, axs = plt.subplots(1, len(models), figsize=(4.5 * len(models), 0.38 * len(order) + 1.5), sharey=True, squeeze=False)
for a, m in zip(axs[0], models):
    g = rec_df[rec_df.model == m].set_index('kondisi').reindex(order)
    y = np.arange(len(order))
    a.errorbar(g.recall, y, xerr=[g.recall - g.lo, g.hi - g.recall], fmt='o', capsize=3)
    a.axvline(g.loc['TEST ASLI (test_sub)', 'recall'], color='gray', ls='--', lw=1)
    a.set_title(m, fontsize=10); a.set_xlabel(f'recall (IoU≥0.5, skor≥{REC_SCORE_THR})'); a.grid(alpha=.3); a.set_xlim(0, 1)
axs[0][0].set_yticks(np.arange(len(order)))
axs[0][0].set_yticklabels([f"{k} (n={len(groups[k]) if k in groups else rec_df[rec_df.kondisi == k].n_foto.iloc[0]})" for k in order])
axs[0][0].invert_yaxis()
plt.suptitle('Recall per kondisi, CI 95% bootstrap per foto (garis abu = test asli)'); plt.tight_layout()
plt.savefig(f"{OUT_ROOT}/figures/recall_per_kondisi.png", dpi=150, bbox_inches='tight'); plt.show()

## 14. Kandidat 5 foto gagal
AP50 dihitung per foto untuk tiap model (fraksi 100%), lalu diurutkan dari rata-rata terendah. Pilih 5 foto dari daftar ini untuk dianalisis di laporan. Pilih yang **penyebabnya beragam** (misalnya gelap, pantulan, oklusi), jangan 5 foto dengan masalah yang sama.

Di visualisasi: kiri = GT, kanan = prediksi dengan skor ≥ 0.5.

In [ ]:
rows = []
for run in F100:
    preds = json.load(open(f"{OUT_ROOT}/preds/{run}__test_custom.json"))
    for i in img_df.image_id:
        m = eval_subset(CUSTOM_GT, preds, [i], 'img')
        rows.append(dict(image_id=i, model=meta(run)[0], AP50=m['AP50'], AP=m['AP']))
per_img = pd.DataFrame(rows).pivot_table(index='image_id', columns='model', values='AP50')
per_img['mean_AP50'] = per_img.mean(1)
per_img = per_img.join(img_df.set_index('image_id')[['file', 'kondisi_utama', 'kondisi_tambahan']]).sort_values('mean_AP50')
display(per_img.head(12).round(3))
per_img.to_csv(f"{OUT_ROOT}/per_image_ap50_f100.csv")
WORST = [int(i) for i in per_img.index[:5]]

In [ ]:
# Ganti WORST dengan image_id pilihanmu kalau ingin 5 foto dengan penyebab yang lebih beragam
for run in F100:
    print('=' * 30, run)
    show_preds(run, 'test_custom', CUSTOM_GT, CUSTOM_IMG_DIR, img_ids=WORST, score_thr=0.5)

## 15. Analisis sensitivitas: bagaimana kalau kelas permukaan ikut dinilai?
Hasil utama (Bagian 12) **hanya memakai kelas objek**, karena label dinding/lantai/plafon di test custom berasal dari auto-label yang tidak dikurasi dan tidak lengkap. Bagian ini mengukur **seberapa besar pilihan itu mengubah kesimpulan**. Varian di bawah dilaporkan **berdampingan** dengan hasil utama, **bukan** dipilih karena AP-nya lebih tinggi.

| Varian | GT test custom |
|---|---|
| `objek` | hasil utama: 42 kelas objek |
| `A_permukaan` | + kelas permukaan, `ceiling` → `ceiling-merged` (nama sesuai dataset asli) |
| `B_permukaan_wall_rename` | A + `wall` → `wall-other-merged` (nama yang dipakai model untuk dinding polos, lihat Bagian 14) |

Di tiap varian, `test_sub` dibentuk ulang dengan kelas yang sama. Prediksi memakai yang sudah tersimpan, jadi tidak ada inferensi ulang. Tabel cakupan menunjukkan **berapa foto yang benar-benar diberi label** tiap kelas permukaan. Kalau suatu kelas terlihat di banyak foto tetapi hanya dilabel di sebagian, prediksi model yang benar di foto lain akan dihitung sebagai *false positive*.

In [ ]:
SURF_NAMES = {'wall', 'wall-brick', 'wall-other-merged', 'wall-stone', 'wall-tile', 'wall-wood',
              'floor-other-merged', 'floor-wood', 'ceiling-merged', 'st-'}
surf_hits = sorted(glob.glob(f"{INPUT_ROOT}/**/test_custom_surface_clean.json", recursive=True))
if not surf_hits:
    print("test_custom_surface_clean.json tidak ada di input -> Bagian 15 dilewati")
else:
    main_imgs = {im['id']: im for im in json.load(open(CUSTOM_GT))['images']}   # ukuran sesudah resize (10.1)

    def build_variant(tag, rename):
        js = json.load(open(surf_hits[0]))
        assert {im['id'] for im in js['images']} == set(main_imgs), "id gambar varian beda dengan test custom utama"
        js['images'] = [main_imgs[im['id']] for im in js['images']]
        for c in js['categories']: c['name'] = rename.get(c['name'], c['name'])
        for a in js['annotations']:
            s = scale[a['image_id']]
            a['segmentation'] = [[v * s for v in p] for p in a['segmentation']]
            a['bbox'] = [v * s for v in a['bbox']]; a['area'] = a['area'] * s * s
        out, dropped = clean_split(js)
        gt = f"{CLEAN_DIR}/test_custom_{tag}.json"; json.dump(out, open(gt, 'w'))
        cats_v = {a['category_id'] for a in out['annotations']}
        sub = f"{CLEAN_DIR}/test_sub_{tag}.json"
        json.dump(dict(tj, annotations=[a for a in tj['annotations'] if a['category_id'] in cats_v]), open(sub, 'w'))
        print(f"{tag}: {len(out['annotations'])} instance, {len(cats_v)} kelas, dibuang: {dropped or '-'}")
        return gt, sub, out

    VARIANTS = {'objek': (CUSTOM_GT, TEST_SUB_JSON, json.load(open(CUSTOM_GT)))}
    VARIANTS['A_permukaan'] = build_variant('A_permukaan', {})
    VARIANTS['B_permukaan_wall_rename'] = build_variant('B_permukaan_wall_rename', {'wall': 'wall-other-merged'})

    # cakupan label permukaan: di berapa foto kelas itu dilabel (test custom) vs muncul (test asli)
    n_test_imgs = len(tj['images'])
    cov = []
    for tag in ['A_permukaan', 'B_permukaan_wall_rename']:
        out = VARIANTS[tag][2]
        for cid in sorted({a['category_id'] for a in out['annotations']}):
            nm = cats_by_id[cid]['name']
            if nm not in SURF_NAMES: continue
            cov.append(dict(varian=tag, kelas=nm,
                            foto_custom=f"{len({a['image_id'] for a in out['annotations'] if a['category_id'] == cid})}/{len(main_imgs)}",
                            foto_test_asli=f"{len({a['image_id'] for a in tj['annotations'] if a['category_id'] == cid})}/{n_test_imgs}"))
    display(pd.DataFrame(cov))

    rows, pcs_rows = [], []
    for run in F100:
        name, _ = meta(run)
        pc = json.load(open(f"{OUT_ROOT}/preds/{run}__test_custom.json")); pt = test_preds_of(run)
        for tag, (gt, sub, _) in VARIANTS.items():
            boots = {}
            for split, gtj, preds in [('test_sub', sub, pt), ('test_custom', gt, pc)]:
                E = coco_evalimgs(gtj, preds); recs = per_image_records(E)
                boots[split] = (ap_from_records(recs, E.params.imgIds), bootstrap_ap(recs, E.params.imgIds))
            (fs, bs), (fc, bc) = boots['test_sub'], boots['test_custom']
            lo, hi = ci_basic(fc[0] - fs[0], bc[:, 0] - bs[:, 0])
            rows.append(dict(model=name, varian=tag, AP_test_sub=fs[0], AP_test_custom=fc[0], drop=fc[0] - fs[0],
                             drop_lo=lo, drop_hi=hi, signifikan='ya (turun)' if hi < 0 else ('ya (naik)' if lo > 0 else 'tidak'),
                             AP50_test_sub=fs[1], AP50_test_custom=fc[1]))
            if tag != 'objek':
                ms, mc = coco_eval_segm(sub, pt, verbose=False), coco_eval_segm(gt, pc, verbose=False)
                for nm in sorted(SURF_NAMES):
                    if nm in mc['per_class'] and not np.isnan(mc['per_class'][nm]):
                        pcs_rows.append(dict(model=name, varian=tag, kelas=nm,
                                             AP_test_sub=ms['per_class'].get(nm, np.nan), AP_test_custom=mc['per_class'][nm]))
        print(f"✓ {run}")
    sens = pd.DataFrame(rows)
    display(sens.round(4))
    sens.to_csv(f"{OUT_ROOT}/sensitivitas_kelas_permukaan_f100.csv", index=False)
    print("AP per kelas permukaan (fraksi 100%):")
    display(pd.DataFrame(pcs_rows).pivot_table(index=['varian', 'kelas'], columns='model',
                                               values=['AP_test_sub', 'AP_test_custom']).round(3))

## 16. Eksperimen: apakah objek kecil gagal karena resolusi input?

**Pengamatan yang mau dijelaskan.** Di test custom, objek kecil seperti `light`, `plug`, dan `cup` sering tidak terdeteksi (lihat Bagian 12 per kelas dan Bagian 14). Ketiga model dilatih dan diuji dengan sisi terpanjang **640 px**, padahal foto HP aslinya sekitar 4000 px. Jadi objek selebar 60 px di foto asli hanya tinggal sekitar 10 px di input model.

**Hipotesis H.** Kegagalan objek kecil disebabkan oleh resolusi input yang terlalu rendah.

**Prediksi, ditulis sebelum eksperimen dijalankan:**
- Kalau H benar: recall objek **kecil** naik jelas dari 640 ke 1024/1280, yaitu CI selisih berpasangan tidak memuat 0. Recall objek **besar** tidak ikut naik, dan bisa saja turun, karena model hanya pernah melihat objek pada skala input 640 (*train–test scale mismatch*).
- Kalau recall objek kecil **tidak** naik: H ditolak. Penjelasan lain yang lebih mungkin adalah sedikitnya contoh kelas itu di data latih, atau perbedaan gaya anotasi.

**Desain eksperimen**
- Bobot fraksi 100% dari ketiga model. **Tanpa training ulang**, hanya resolusi inferensi yang diubah: `EXP_SIZES`.
- Ukuran objek diukur **pada skala input 640**, memakai batas COCO: kecil < 32², sedang 32²–96², besar > 96² piksel. Dengan begitu kelompok "kecil" berisi objek yang sama di semua resolusi.
- Metrik utama: **recall** (IoU ≥ 0.5, skor ≥ `REC_SCORE_THR`) per kelompok ukuran. Perubahannya dibandingkan dengan 640 lewat **bootstrap berpasangan per foto**: foto yang sama di-resample untuk kedua resolusi.
- Sanity check: AP hasil run 640 di `test_custom` harus sama dengan hasil Bagian 11.
- **Dua versi gambar:**
  - `test_custom_full` (**uji utama**): foto asli sekitar 4032 px dengan GT di resolusi asli. Pada versi ini, inferensi di 1024/1280 benar-benar melihat **detail lebih banyak**.
  - `test_custom`: foto yang sudah dikecilkan ke resolusi test asli (Bagian 10.1, sekitar 800 px). Pada versi ini, 1024/1280 hanya **memperbesar foto** tanpa menambah informasi, jadi yang diuji adalah efek *skala objek*, bukan detail. Versi ini dipertahankan sebagai **pembanding**. Kalau kedua versi memberi hasil serupa, berarti yang membantu adalah skala objek, bukan detail tambahan.
- Hanya test custom yang diuji. Test asli tidak dipakai, karena berisi 359 foto (sebagian berukuran sangat besar, yang dulu memicu OOM Mask2Former), sehingga inferensi ulang di 3 resolusi terlalu lama. Set `EXP_ON_TEST = True` kalau ingin menambahkan test asli sebagai kontrol in-domain.

In [ ]:
EXP_SIZES   = [640, 1024, 1280]      # kelipatan 32 (syarat YOLO)
EXP_ON_TEST = False
EXP_DIR = f"{OUT_ROOT}/preds/exp_resolusi"; os.makedirs(EXP_DIR, exist_ok=True)

@torch.no_grad()
def predict_m2f_at(model, gt_json, img_dir, S):
    # sama dengan predict_m2f, tapi resolusi input S dioper eksplisit (default argumen S terkunci di IMG_SIZE)
    model.eval()
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    res = []
    for img in coco.dataset['images']:
        pil = load_image(os.path.join(img_dir, img['file_name']))
        pv, pm, (nh, nw) = m2f_preprocess(pil, S=S)
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            out = model(pixel_values=pv[None].to(DEVICE), pixel_mask=pm[None].to(DEVICE))
        masks, labels, scores = m2f_postprocess(out.class_queries_logits[0], out.masks_queries_logits[0],
                                                nh, nw, pil.size[1], pil.size[0], S=S)
        for m, l, s in zip(masks, labels, scores):
            if m.any():
                res.append({'image_id': img['id'], 'category_id': CAT_IDS[int(l)],
                            'segmentation': mask_to_rle(m), 'score': float(s)})
    return res

from torchvision.models.detection.roi_heads import paste_masks_in_image
from torchvision.models.detection.transform import resize_boxes

@torch.no_grad()
def predict_maskrcnn_at(model, gt_json, img_dir, S, chunk=10):
    # Untuk foto besar (12 MP): postprocess bawaan torchvision menempel 100 mask float di resolusi asli sekaligus
    # (~4.5 GB) -> OOM di S>=1024. Di sini forward dijalankan per tahap (transform -> backbone -> RPN -> ROI heads),
    # persis seperti GeneralizedRCNN.forward, lalu langkah paste yang SAMA (paste_masks_in_image) dijalankan per
    # `chunk` mask. Hasilnya identik dengan cara bawaan, hanya memorinya lebih kecil.
    model.eval(); model.roi_heads.score_thresh = 0.001; model.roi_heads.detections_per_img = 100
    model.transform.min_size, model.transform.max_size = (S,), S
    with contextlib.redirect_stdout(io.StringIO()):
        coco = COCO(gt_json)
    res = []
    for img in coco.dataset['images']:
        x = TF.to_tensor(load_image(os.path.join(img_dir, img['file_name']))).to(DEVICE)
        H, W = x.shape[-2:]
        with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            images, _ = model.transform([x])
            feats = model.backbone(images.tensors)
            props, _ = model.rpn(images, feats)
            det = model.roi_heads(feats, props, images.image_sizes)[0][0]
        boxes = resize_boxes(det['boxes'].float(), images.image_sizes[0], (H, W))
        probs, labels, scores = det['masks'].float(), det['labels'].cpu().numpy(), det['scores'].float().cpu().numpy()
        del x, images, feats, props
        for c in range(0, len(probs), chunk):
            m = (paste_masks_in_image(probs[c:c + chunk], boxes[c:c + chunk], (H, W))[:, 0] > 0.5).cpu().numpy()
            for mm, l, s in zip(m, labels[c:c + chunk], scores[c:c + chunk]):
                res.append({'image_id': img['id'], 'category_id': CAT_IDS[int(l) - 1],
                            'segmentation': mask_to_rle(mm), 'score': float(s)})
    return res

def exp_path(run, split, S):
    # Mask R-CNN di foto resolusi asli memakai metode chunk (lihat di atas) -> nama file sendiri
    tag = '_chunk' if run.startswith('maskrcnn') and split == 'test_custom_full' else ''
    return f"{EXP_DIR}/{run}__{split}__S{S}{tag}.json"

def predict_at(run, gt_json, img_dir, S, chunked=False):
    global IMG_SIZE
    w = f"{TRAIN_OUT}/runs/{run}"
    if run.startswith('yolo'):
        old, IMG_SIZE = IMG_SIZE, S                     # predict_yolo membaca IMG_SIZE global saat dipanggil
        try:
            wt = f"{w}/weights/best.pt" if os.path.exists(f"{w}/weights/best.pt") else f"{w}/weights/last.pt"
            return predict_yolo(wt, gt_json, img_dir)
        finally:
            IMG_SIZE = old
    wt = f"{w}/best.pt" if os.path.exists(f"{w}/best.pt") else f"{w}/last.pt"
    if run.startswith('maskrcnn'):
        model = build_maskrcnn(pretrained=False); model.load_state_dict(torch.load(wt, map_location=DEVICE))
        model.transform.min_size, model.transform.max_size = (S,), S     # sisi terpanjang = S
        try: return predict_maskrcnn_at(model, gt_json, img_dir, S) if chunked else predict_maskrcnn(model, gt_json, img_dir)
        finally: del model
    model = build_m2f(pretrained=False); model.load_state_dict(torch.load(wt, map_location=DEVICE))
    try: return predict_m2f_at(model, gt_json, img_dir, S)
    finally: del model

# path input bisa berubah kalau dataset Kaggle di-update (New Version) di tengah sesi -> cari ulang
if not os.path.exists(CUSTOM_JSON):
    CUSTOM_JSON = find_one(f"{INPUT_ROOT}/**/test_custom_clean.json", 'test_custom_clean.json')
if not os.path.isdir(CUSTOM_SRC_DIR):
    CUSTOM_SRC_DIR = os.path.dirname(find_one(f"{INPUT_ROOT}/**/{glob.escape(first)}", 'gambar test custom'))
print("anotasi:", CUSTOM_JSON, "| gambar asli:", CUSTOM_SRC_DIR)

# GT resolusi asli: anotasi custom TANPA resize (10.1 mengecilkan ke resolusi test asli)
out_full, _ = clean_split(json.load(open(CUSTOM_JSON)))
CUSTOM_GT_FULL = f"{CLEAN_DIR}/test_custom_full.json"; json.dump(out_full, open(CUSTOM_GT_FULL, 'w'))
print("resolusi GT full:", sorted({max(i['width'], i['height']) for i in out_full['images']}),
      "| resolusi test_custom (10.1):", TARGET_LONG)

exp_sets = [('test_custom_full', CUSTOM_GT_FULL, CUSTOM_SRC_DIR), ('test_custom', CUSTOM_GT, CUSTOM_IMG_DIR)] + \
           ([('test_sub', TEST_SUB_JSON, IMG_DIRS['test'])] if EXP_ON_TEST else [])
for run in F100:
    name, _ = meta(run)
    for split, gtj, idir in exp_sets:
        for S in EXP_SIZES:
            p = exp_path(run, split, S)
            if os.path.exists(p): continue
            try:
                with GPUTimer(f"{run}__exp_res_S{S}_{split}", name, note=f'eksperimen resolusi S={S}'):
                    preds = predict_at(run, gtj, idir, S, chunked=p.endswith('_chunk.json'))
                json.dump(preds, open(p, 'w'))
                print(f"  {run} {split} S={S}: {len(preds)} prediksi")
            except Exception as e:
                traceback.print_exc(); log_note(f"GAGAL eksperimen resolusi {run} S={S}: {type(e).__name__}: {str(e)[:200]}")
            finally:
                gc.collect(); torch.cuda.empty_cache()

# sanity check: S=640 harus sama dengan hasil Bagian 11
for run in F100:
    ref = json.load(open(f"{OUT_ROOT}/results/{run}__test_custom.json"))['AP']
    p = f"{EXP_DIR}/{run}__test_custom__S640.json"
    if os.path.exists(p):
        ap = coco_eval_segm(CUSTOM_GT, json.load(open(p)), verbose=False)['AP']
        print(f"sanity {run}: AP S640 = {ap:.4f} | Bagian 11 = {ref:.4f} | {'OK' if abs(ap - ref) < 2e-3 else 'BEDA — cek!'}")
    pf = f"{EXP_DIR}/{run}__test_custom_full__S640.json"
    if os.path.exists(pf):   # tidak harus identik: jalur resize & resolusi mask/GT berbeda, tapi harus dekat
        apf = coco_eval_segm(CUSTOM_GT_FULL, json.load(open(pf)), verbose=False)['AP']
        print(f"   versi full S640 = {apf:.4f} (selisih {apf - ref:+.4f}; wajar bila kecil, < ~0.01)")
    pc = exp_path(run, 'test_custom_full', 640)
    if pc.endswith('_chunk.json') and os.path.exists(pc) and os.path.exists(pf):
        apc = coco_eval_segm(CUSTOM_GT_FULL, json.load(open(pc)), verbose=False)['AP']
        print(f"   validasi metode chunk Mask R-CNN @640: chunk = {apc:.4f} vs bawaan torchvision = {apf:.4f} "
              f"(selisih {apc - apf:+.4f}; harus sangat kecil)")

In [ ]:
def ann_hits(gt_json, preds, cat_ids):
    # {ann_id: True/False}: GT ditemukan (kelas benar, IoU>=0.5, skor>=REC_SCORE_THR)
    p = [d for d in preds if d['score'] >= REC_SCORE_THR and d['category_id'] in cat_ids]
    hit = {}
    if p:
        E = coco_evalimgs(gt_json, p, iou_thrs=[0.5])
        for e in E.evalImgs:
            if e is None: continue
            for gid, m, ig in zip(e['gtIds'], e['gtMatches'][0], e['gtIgnore']):
                if not ig: hit[gid] = bool(m != 0)
    return hit

def size_table(gt_json):
    js = json.load(open(gt_json)); dims = {im['id']: (im['width'], im['height']) for im in js['images']}
    rows = []
    for a in js['annotations']:
        if a.get('iscrowd', 0) or a['category_id'] not in CUSTOM_CATS: continue
        a640 = a['area'] * (640 / max(dims[a['image_id']])) ** 2         # luas pada skala input 640
        rows.append(dict(ann_id=a['id'], image_id=a['image_id'], cls=cats_by_id[a['category_id']]['name'], area640=a640,
                         ukuran='kecil' if a640 < 32 ** 2 else ('sedang' if a640 < 96 ** 2 else 'besar')))
    return pd.DataFrame(rows)

def paired_boot_diff(df, col_a, col_b, B=BOOT_B, seed=BOOT_SEED):
    # recall(col_b) - recall(col_a) dengan resample FOTO (berpasangan: foto sama untuk kedua kolom)
    g = df.groupby('image_id').agg(n=('ann_id', 'size'), a=(col_a, 'sum'), b=(col_b, 'sum'))
    n, a, b = g.n.values, g.a.values, g.b.values
    rng = np.random.default_rng(seed); diffs = []
    for _ in range(B):
        s = rng.integers(0, len(g), len(g))
        if n[s].sum(): diffs.append(b[s].sum() / n[s].sum() - a[s].sum() / n[s].sum())
    d = b.sum() / n.sum() - a.sum() / n.sum()
    return (d, *ci_basic(d, np.array(diffs))) if len(g) >= 3 else (d, np.nan, np.nan)

for split, gtj, _ in exp_sets:
    st = size_table(gtj)
    print(f"[{split}] jumlah objek per ukuran (skala 640):", st.ukuran.value_counts().to_dict())
    rows, cls_rows, aps = [], [], []
    for run in F100:
        name, _ = meta(run)
        for S in EXP_SIZES:
            p = exp_path(run, split, S)
            if not os.path.exists(p): continue
            preds = json.load(open(p)); h = ann_hits(gtj, preds, set(CUSTOM_CATS))
            st[f"{name}|{S}"] = st.ann_id.map(lambda i: h.get(i, False)).astype(int)
            m = coco_eval_segm(gtj, preds, verbose=False); aps.append(dict(model=name, S=S, AP=m['AP'], AP50=m['AP50']))
        for S in EXP_SIZES[1:]:
            a, b = f"{name}|{EXP_SIZES[0]}", f"{name}|{S}"
            if a not in st or b not in st: continue
            for uk in ['kecil', 'sedang', 'besar', 'semua']:
                sub = st if uk == 'semua' else st[st.ukuran == uk]
                d, lo, hi = paired_boot_diff(sub, a, b)
                rows.append(dict(model=name, ukuran=uk, n_objek=len(sub), S_vs=f"{S} vs {EXP_SIZES[0]}",
                                 recall_640=sub[a].mean(), recall_S=sub[b].mean(), selisih=d, lo=lo, hi=hi,
                                 signifikan='ya (naik)' if lo > 0 else ('ya (turun)' if hi < 0 else 'tidak')))
        for c in ['light', 'plug', 'cup', 'bottle', 'cell phone', 'paper', 'chair', 'person']:
            sub = st[st.cls == c]
            if len(sub): cls_rows.append({'model': name, 'kelas': c, 'n': len(sub),
                                          **{f"recall_S{S}": sub[f'{name}|{S}'].mean() for S in EXP_SIZES if f'{name}|{S}' in st}})
    if not aps:
        print(f"[{split}] belum ada prediksi eksperimen -> dilewati"); continue
    exp_df = pd.DataFrame(rows)
    print("\nPerubahan recall vs S=640 (bootstrap berpasangan per foto, CI 95%):"); display(exp_df.round(3))
    print("AP keseluruhan per resolusi:"); display(pd.DataFrame(aps).pivot_table(index='model', columns='S', values=['AP', 'AP50']).round(4))
    print("Recall per kelas yang dicurigai (kecil) + pembanding (besar):"); display(pd.DataFrame(cls_rows).round(3))
    exp_df.to_csv(f"{OUT_ROOT}/eksperimen_resolusi_{split}.csv", index=False)

    fig, axs = plt.subplots(1, len(F100), figsize=(4.5 * len(F100), 3.8), sharey=True, squeeze=False)
    for ax, run in zip(axs[0], F100):
        name, _ = meta(run)
        for uk, mk in [('kecil', 'o'), ('sedang', 's'), ('besar', '^')]:
            sub = st[st.ukuran == uk]
            ys = [sub[f'{name}|{S}'].mean() for S in EXP_SIZES if f'{name}|{S}' in st]
            ax.plot(EXP_SIZES[:len(ys)], ys, mk + '-', label=f"{uk} (n={len(sub)})")
        ax.set_title(name, fontsize=10); ax.set_xlabel('resolusi inferensi (sisi terpanjang)'); ax.set_xticks(EXP_SIZES); ax.grid(alpha=.3)
    axs[0][0].set_ylabel(f'recall (IoU≥0.5, skor≥{REC_SCORE_THR})'); axs[0][0].legend(fontsize=8)
    plt.suptitle(f'Eksperimen resolusi — {split} (model dilatih di 640)'); plt.tight_layout()
    plt.savefig(f"{OUT_ROOT}/figures/eksperimen_resolusi_{split}.png", dpi=150); plt.show()